# ComfyUI — klein-9B Skin Enhancer (clean rebuild)

**Before running:**
1. Runtime → Change runtime type → **GPU (A100/L4)**.
2. Add your **HF_TOKEN** in the 🔑 Secrets tab (and toggle notebook access ON).
3. Accept the klein-9B license once at https://huggingface.co/black-forest-labs/FLUX.2-klein-9B → 'Agree and access'.

Then run cells **1 → 4** top to bottom. Everything installs into `/content/ComfyUI` (local disk).
On a fresh runtime, just re-run all four — no manual file copying.

_Cell 4 stays running and embeds the ComfyUI UI inline — scroll down inside its output to use it._

In [ ]:
# CELL 1 — Install ComfyUI + Manager + dependencies
import os
COMFY = '/content/ComfyUI'
if not os.path.isdir(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY}
%cd {COMFY}
!pip install -q -r requirements.txt           # installs comfy_aimdo etc. (fixes the old ModuleNotFound)
if not os.path.isdir(f'{COMFY}/custom_nodes/comfyui-manager'):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager {COMFY}/custom_nodes/comfyui-manager
print('✅ ComfyUI + Manager ready')

In [ ]:
# CELL 2 — Custom nodes the workflow needs
import os, subprocess
CN = '/content/ComfyUI/custom_nodes'
repos = {
    'ComfyUI_essentials':            'https://github.com/cubiq/ComfyUI_essentials',
    'comfyui_face_parsing':          'https://github.com/Ryuukeisyou/comfyui_face_parsing',
    'ComfyUI_LayerStyle':            'https://github.com/chflame163/ComfyUI_LayerStyle',
    'ComfyUI-Impact-Pack':           'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
    'ComfyUI-KJNodes':               'https://github.com/kijai/ComfyUI-KJNodes',
    'ComfyUI-SeedVR2_VideoUpscaler': 'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler',
}
for folder, url in repos.items():
    p = os.path.join(CN, folder)
    if os.path.isdir(p): print('⏭️', folder)
    else: print('⬇️', folder); subprocess.run(['git','clone','--depth','1',url,p], check=False)
for folder in repos:
    req = os.path.join(CN, folder, 'requirements.txt')
    if os.path.exists(req): subprocess.run(['pip','install','-q','-r',req], check=False)
print('✅ custom nodes ready (face_parsing + seedvr2 download their model weights on first run)')

In [ ]:
# CELL 3 — Models: klein-9B + qwen_3_8b + VAE + the two klein LoRAs
import os, shutil
from huggingface_hub import hf_hub_download, list_repo_files
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
C = '/content/ComfyUI'
for d in ['diffusion_models','text_encoders','vae','loras']:
    os.makedirs(f'{C}/models/{d}', exist_ok=True)

def pull(repo, fname, dest, token=None):
    p = hf_hub_download(repo, fname, local_dir=dest, token=token)
    flat = os.path.join(dest, os.path.basename(fname))
    if p != flat: shutil.move(p, flat)
    # remove any leftover split_files/ subdir
    sub = os.path.join(dest, fname.split('/')[0])
    if '/' in fname and os.path.isdir(sub): shutil.rmtree(sub, ignore_errors=True)
    return flat

# klein-9B diffusion model (GATED — needs HF_TOKEN + accepted license)
pull('black-forest-labs/FLUX.2-klein-9B', 'flux-2-klein-9b.safetensors', f'{C}/models/diffusion_models', HF_TOKEN)
# qwen_3_8b text encoder
pull('Comfy-Org/flux2-klein-9B', 'split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors', f'{C}/models/text_encoders', HF_TOKEN)
# VAE
pull('Comfy-Org/flux2-dev', 'split_files/vae/flux2-vae.safetensors', f'{C}/models/vae')

# --- the two klein-native LoRAs, renamed to the names the v5 workflow expects ---
def grab_lora(repo, out_name):
    sf = [f for f in list_repo_files(repo) if f.endswith('.safetensors')][0]
    src = hf_hub_download(repo, sf, local_dir=f'{C}/models/loras')
    dst = f'{C}/models/loras/{out_name}'
    if src != dst: shutil.move(src, dst)
    print('✅ LoRA:', out_name)
grab_lora('dx8152/Flux2-Klein-9B-Enhanced-Details', 'klein-enhanced-details.safetensors')
grab_lora('linoyts/Flux2-Klein-Delight-LoRA',       'klein-delight.safetensors')

!ls -lh {C}/models/diffusion_models {C}/models/text_encoders {C}/models/vae {C}/models/loras

In [ ]:
# CELL 4 — Launch ComfyUI (embedded inline). KEEP RUNNING. Scroll down in this cell's output for the UI.
import threading, time, socket
%cd /content/ComfyUI
def serve(port):
    while True:
        time.sleep(0.5)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(('127.0.0.1', port)) == 0: s.close(); break
        s.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1024)
threading.Thread(target=serve, daemon=True, args=(8188,)).start()
!python main.py --dont-print-server --enable-cors-header